<a href="https://colab.research.google.com/github/saitejamudapalli/Project-HealthCare-Provider-Analysis/blob/main/SILVER_LAYER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CODE FOR SILVER_LAYER

from google.cloud import bigquery
from google.oauth2 import service_account
import pandas as pd
import numpy as np

# ===== CONFIG =====
PROJECT_ID = "even-blueprint-441418-p2"
SOURCE_DATASET = "BRONZE_LAYER"
SOURCE_TABLE = "PATIENTS_BRONZE"
TARGET_DATASET = "SILVER_LAYER"
TARGET_TABLE = "PATIENTS_SILVER"
KEY_PATH = "/content/even-blueprint-441418-p2-043f8a9d855b.json(KEY).json"

# ===== AUTH & READ BRONZE =====
credentials = service_account.Credentials.from_service_account_file(KEY_PATH)
client = bigquery.Client(credentials=credentials, project=PROJECT_ID)

query = f"SELECT * FROM `{PROJECT_ID}.{SOURCE_DATASET}.{SOURCE_TABLE}`"
df_bronze = client.query(query).to_dataframe()

# ===== CLEANING & STANDARDIZATION =====
df_silver = df_bronze.copy()

# 1️⃣ Normalize column names
df_silver.columns = [c.strip().lower().replace(" ", "_") for c in df_silver.columns]

# 2️⃣ Replace 'nan', 'NaN', 'NaT' strings with np.nan for detection
df_silver = df_silver.replace(
    to_replace=["nan", "NaN", "NaT"],
    value=np.nan
)

# 3️⃣ Trim string columns safely
for c in df_silver.select_dtypes(include=["object"]).columns:
    df_silver[c] = df_silver[c].astype(str).str.strip()

# 4️⃣ Convert numeric-like columns safely
for c in df_silver.columns:
    try:
        coerced = pd.to_numeric(df_silver[c], errors="ignore")  # don’t coerce, keep None
        df_silver[c] = coerced
    except Exception:
        pass

# 5️⃣ Drop duplicates
df_silver = df_silver.drop_duplicates()

# 6️⃣ 🚫 Remove only rows containing NaN (not None)
# Keep rows that have Python None, but remove those with np.nan
mask_nan_rows = df_silver.apply(lambda row: row.isna().any(), axis=1)
df_silver = df_silver[~mask_nan_rows].reset_index(drop=True)

# 7️⃣ Upload to BigQuery
table_ref = f"{PROJECT_ID}.{TARGET_DATASET}.{TARGET_TABLE}"
job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
client.delete_table(table_ref, not_found_ok=True)
job = client.load_table_from_dataframe(df_silver, table_ref, job_config=job_config)
job.result()

print(f"✅ Silver layer loaded successfully to {table_ref}")
print(f"💡 Rows after removing NaN only (None kept): {len(df_silver)}")
